## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

If SendGrid gives you problems, see the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

Please set up an account - it's free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.


In [1]:
%load_ext autoreload
%autoreload 2

In [97]:
from dotenv import load_dotenv
from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

import sys
sys.path.append('../my-utils')
from my_mailer import send_email as mailer_send_email
from my_mailer import send_html_email as mailer_send_html_email



In [79]:
# test sending myself an email
await mailer_send_email(
    to ="3liotm@gmail.com",
    subject="Test Email",
    body="This is a test email sent from the async email sender."
)

Email sent successfully to 3liotm@gmail.com


In [33]:
load_dotenv(override=True)

True

In [ ]:
# Let's just check emails are working for you

# def send_test_email():
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com")  # Change to your verified sender
#     to_email = To("ed.donner@gmail.com")  # Change to your recipient
#     content = Content("text/plain", "This is an important test email")
#     mail = Mail(from_email, to_email, "Test email", content).get()
#     response = sg.client.mail.send.post(request_body=mail)
#     print(response.status_code)

# send_test_email()

### Did you receive the test email

If you get a 202, then you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [28]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

instructions4 = "You are golum, the mythical creature from lord of the rings. \
you are tricksy and untrusting.  Somehow you got a job as a sales agent for ComplAI, \
probably because they are desperate.  You write cold emails in a suspicious and untrusting tone."

In [ ]:
# sales_agent1 = Agent(
#         name="Professional Sales Agent",
#         instructions=instructions1,
#         model=grok4_model
# )

# sales_agent2 = Agent(
#         name="Engaging Sales Agent",
#         instructions=instructions2,
#         model=grok4_model
# )

# sales_agent3 = Agent(
#         name="Busy Sales Agent",
#         instructions=instructions3,
#         model=grok4_model
# )

Using model: grok-4


In [ ]:
grok_client = AsyncOpenAI(
    base_url=os.getenv("GROK_BASE_URL"),  # Should be "https://api.x.ai/v1" or similar
    api_key=os.getenv("GROK_API_KEY")
)

In [15]:
print("GEMINI_URL:", os.getenv("GEMINI_BASE_URL"))
print("GEMINI_API_KEY:", os.getenv("GEMINI_API_KEY")[:4] + "...")

gemini_client = AsyncOpenAI(
    base_url=os.getenv("GEMINI_BASE_URL"),  # Should be "https://gemini-api.labs.google.com/v1" or similar
    api_key=os.getenv("GEMINI_API_KEY")
)

GEMINI_URL: https://generativelanguage.googleapis.com/v1beta/openai/
GEMINI_API_KEY: AIza...


In [ ]:
grok4_model = os.getenv("GROK4F_MODEL")

grok_sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)

grok_sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)

grok_sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,       
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)

grok_sales_agent4 = Agent(
    name="Gollum Sales Agent",
    instructions=instructions4,       
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)


In [ ]:
# An example of running and streaming the output from one of the sales agents
result = Runner.run_streamed(grok_sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC2 Compliance with AI-Powered Efficiency

Dear [Prospect's Name],

I hope this email finds you well. My name is Alex Rivera, and I'm a sales representative at ComplAI, where we specialize in AI-driven SaaS solutions designed to simplify SOC2 compliance and audit preparation for growing tech companies like yours.

In today's regulatory landscape, maintaining SOC2 compliance can be a time-consuming and resource-intensive process—often involving manual documentation, endless checklists, and the risk of oversights that could lead to audit failures. At ComplAI, we've developed an intuitive platform that automates these tasks, leveraging advanced AI to generate compliant policies, track controls in real-time, and prepare audit-ready reports with minimal effort.

Our clients, including innovative SaaS providers and fintech startups, have reduced their compliance prep time by up to 70% while ensuring accuracy and peace of mind. I'd love to show you how ComplAI can do

In [30]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(grok_sales_agent1, message),
        Runner.run(grok_sales_agent2, message),
        Runner.run(grok_sales_agent3, message),
        Runner.run(grok_sales_agent4, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Streamline Your SOC 2 Compliance with AI-Powered Efficiency

Dear [Prospect's Name],

I hope this email finds you well. My name is Alex Rivera, and I'm a sales representative at ComplAI, a leading SaaS platform designed to simplify SOC 2 compliance and audit preparation through advanced AI technology.

In today's regulatory landscape, achieving and maintaining SOC 2 compliance can be a time-consuming and resource-intensive process. Many organizations struggle with manual documentation, risk assessments, and evidence collection, often leading to delays and increased costs during audits.

That's where ComplAI comes in. Our AI-driven tool automates key compliance tasks, including:

- Intelligent risk identification and mitigation recommendations
- Automated evidence gathering and policy generation
- Real-time audit readiness dashboards

Companies like [Fictional Client A] and [Fictional Client B] have reduced their audit preparation time by up to 50% and improved compliance score

In [32]:
# sales_picker = Agent(
#     name="sales_picker",
#     instructions="You pick the best cold sales email from the given options. \
# Imagine you are a customer and pick the one you are most likely to respond to. \
# Do not give an explanation; reply with the selected email only.",
#     model="gpt-4o-mini"
# )

In [89]:
fast_model = os.getenv("GEMINI_FAST_MODEL")
print("Using Gemini Fast model:", fast_model)

sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",      
    model=OpenAIChatCompletionsModel(
        model=fast_model,
        openai_client=gemini_client
    )
)

Using Gemini Fast model: gemini-2.5-flash


In [36]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    # results = await asyncio.gather(
    #     Runner.run(grok_sales_agent1, message),
    #     Runner.run(grok_sales_agent2, message),
    #     Runner.run(grok_sales_agent3, message),
    # )
    # outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: SOC2 Compliance: Because "Wing It" Isn't an Audit Strategy

Dear [Recipient's Name],

Ever feel like SOC2 compliance is that one relative who shows up unannounced, demands your full attention, and leaves you questioning your life choices? Yeah, we've all been there. But what if I told you there's an AI sidekick that turns that audit nightmare into a walk in the park?

I'm [Your Name] from ComplAI, where we're revolutionizing SOC2 prep with our SaaS tool. Powered by cutting-edge AI, it automates evidence collection, gap analysis, and report generation—saving you weeks of manual drudgery and that sinking feeling of "Did I miss something?"

Picture this:
- **Zero spreadsheets from hell**: Our AI scans your systems, flags issues, and even suggests fixes.
- **Audit-ready in record time**: Clients like [Fictional Client Example] cut prep time by 60% and passed with flying colors.
- **Humor included**: Because compliance doesn't have to be as dry as a tax form.

No 

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [37]:
# sales_agent1 = Agent(
#         name="Professional Sales Agent",
#         instructions=instructions1,
#         model="gpt-4o-mini",
# )

# sales_agent2 = Agent(
#         name="Engaging Sales Agent",
#         instructions=instructions2,
#         model="gpt-4o-mini",
# )

# sales_agent3 = Agent(
#         name="Busy Sales Agent",
#         instructions=instructions3,
#         model="gpt-4o-mini",
# )

In [38]:
grok_sales_agent1

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoff_description=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x78f305d0c740>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [ ]:
# @function_tool
# def send_email(body: str):
#     """ Send out an email with the given body to all sales prospects """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com")  # Change to your verified sender
#     to_email = To("ed.donner@gmail.com")  # Change to your recipient
#     content = Content("text/plain", body)
#     mail = Mail(from_email, to_email, "Sales email", content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return {"status": "success"}

In [85]:
@function_tool
async def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    await mailer_send_email(
        to="3liotm@gmail.com",
        subject="Sales email",
        body=body
    )
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [86]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f304677240>, strict_json_schema=True, is_enabled=True)

### And you can also convert an Agent into a tool

In [87]:
tool1 = grok_sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f304677a60>, strict_json_schema=True, is_enabled=True)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [88]:
description = "Write a cold sales email"

tool1 = grok_sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = grok_sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = grok_sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)
tool4 = grok_sales_agent4.as_tool(tool_name="sales_agent4", tool_description=description)

tools = [tool1, tool2, tool3, tool4, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f3047f3e20>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f3047c5c60>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'req

## And now it's time for our Sales Manager - our planning agent

In [91]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all four sales_agent tools to generate four different email drafts. Do not proceed until all four drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


# sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")
sales_manager = Agent(
    name="Sales Manager",
    instructions=instructions, 
    tools=tools,     
    model=OpenAIChatCompletionsModel(
        model=fast_model,
        openai_client=gemini_client
    )
)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

Email sent successfully to 3liotm@gmail.com


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across and the orignal agent is giving up control to the next agent



In [125]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

#subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
#subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

# html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
# html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

# subject_instructions = "You can write a subject for a cold sales email. \
# You are given a message and you need to write a subject for an email that is likely to get a response."

# html_instructions = "You can convert a text email body to an HTML email body. \
# You are given a text email body which might have some markdown \
# and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

# flash_model = os.getenv("GEMINI_FAST_MODEL")

# subject_writer = Agent(
#     name="Email subject writer", 
#     instructions=subject_instructions, 
#     model=OpenAIChatCompletionsModel(
#         model=flash_model,
#         openai_client=gemini_client
#     )
# )
# subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

# html_converter = Agent(
#     name="HTML email body converter", 
#     instructions=html_instructions, 
#     model=OpenAIChatCompletionsModel(
#         model=flash_model,
#         openai_client=gemini_client
#     )
# )
# html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")


subject_writer = Agent(
    name="Email subject writer", 
    instructions=subject_instructions, 
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(
    name="HTML email body converter", 
    instructions=html_instructions, 
    model=OpenAIChatCompletionsModel(
        model=grok4_model,
        openai_client=grok_client
    )
)

html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")


In [126]:
# @function_tool
# def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
#     """ Send out an email with the given subject and HTML body to all sales prospects """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com")  # Change to your verified sender
#     to_email = To("ed.donner@gmail.com")  # Change to your recipient
#     content = Content("text/html", html_body)
#     mail = Mail(from_email, to_email, subject, content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return {"status": "success"}

In [127]:
@function_tool
async def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    await mailer_send_html_email(
        to="3liotm@gmail.com",  # Your email
        subject=subject,
        html_body=html_body
    )
    return {"status": "success"}

In [128]:
tools = [subject_tool, html_tool, send_html_email]

In [129]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f304bd7e20>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f304bd4b80>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', description='Send out an email with the given subject and HTML body to all sale

In [130]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


# emailer_agent = Agent(
#     name="Email Manager",
#     instructions=instructions,
#     tools=tools,
#     model="gpt-4o-mini",
#     handoff_description="Convert an email to HTML and send it")


### Handoff Description

This is how an agent anounces itself to the world about what it can do

In [131]:
emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=OpenAIChatCompletionsModel(
        model='gemini-2.5-flash',
        openai_client=gemini_client
    ),
    handoff_description="Convert an email to HTML and send it")

### Now we have 3 tools and 1 handoff

In [132]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f3047f3e20>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78f3047c5c60>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'requi

In [136]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


# sales_manager = Agent(
#     name="Sales Manager",
#     instructions=sales_manager_instructions,
#     tools=tools,
#     handoffs=handoffs,
#     model="gpt-4o-mini")

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=OpenAIChatCompletionsModel(
        model='gemini-2.5-flash',
        openai_client=gemini_client
    )
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

HTML email sent successfully to 3liotm@gmail.com


### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR (Sales Development Representative) respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

In [134]:
q_answer = '''
Agentic Design Patterns Used:
Planning/Reflection - The Sales Manager agent evaluates multiple email drafts and selects the best one, demonstrating decision-making capability.
Tool Use - Agents can call functions (like send_email) and other agents as tools to accomplish tasks.
Multi-Agent Collaboration - Multiple specialized agents (professional, engaging, busy sales agents) work together, each with different personas.
Orchestration - The Sales Manager acts as an orchestrator, coordinating the work of other agents.
Handoffs - Control passes from the Sales Manager to the Email Manager agent to handle formatting and sending.
The 1 Line That Changed "Workflow" to "Agent":
According to Anthropic's definition, the key difference is dynamic decision-making with tools. The critical line is:
tools = [tool1, tool2, tool3, send_email]
or more specifically when defining the Sales Manager:
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")
Why this matters:
Workflow: Pre-defined, deterministic sequence of steps. The agent follows a fixed path.
Agent (Anthropic's definition): The agent can dynamically decide which tools to use, when to use them, and in what order based on the task at hand.
The moment you give an agent tools that it can choose to invoke based on reasoning (rather than a hardcoded sequence), it becomes an "agent" under Anthropic's definition. The LLM decides the execution path, not the programmer. In earlier cells, you had hardcoded sequences with asyncio.gather() - that's a workflow. But when the Sales Manager can decide which sales agents to call and when, that's agentic behavior!
'''

print(q_answer)


Agentic Design Patterns Used:
Planning/Reflection - The Sales Manager agent evaluates multiple email drafts and selects the best one, demonstrating decision-making capability.
Tool Use - Agents can call functions (like send_email) and other agents as tools to accomplish tasks.
Multi-Agent Collaboration - Multiple specialized agents (professional, engaging, busy sales agents) work together, each with different personas.
Orchestration - The Sales Manager acts as an orchestrator, coordinating the work of other agents.
Handoffs - Control passes from the Sales Manager to the Email Manager agent to handle formatting and sending.
The 1 Line That Changed "Workflow" to "Agent":
According to Anthropic's definition, the key difference is dynamic decision-making with tools. The critical line is:
tools = [tool1, tool2, tool3, send_email]
or more specifically when defining the Sales Manager:
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")
Why

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.